# Python Challenge 3 – Personal Movie Tracker with JSON
A mini-IMDb for the command line. All movies live in `movies.json`; each movie has **title, genre, rating (out of 10)**.

Menu: **Add a movie · View all movies · Search by title or genre · Exit**

Bonus covered: duplicate titles are rejected · output is a clean table · structured data is read/written with `json`.

In [1]:
import json
import os

JSON_FILE = "movies.json"

## JSON file handling
The file stores a list of dictionaries, for example:
```json
[{"title": "Inception", "genre": "Sci-Fi", "rating": 8.8}]
```
A missing or corrupted file is handled gracefully (we start with an empty list).

In [2]:
def load_movies(path=JSON_FILE):
    if not os.path.exists(path):
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else []
    except (json.JSONDecodeError, OSError):
        print("Warning: could not read the movie file, starting with an empty list.")
        return []

def save_movies(movies, path=JSON_FILE):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(movies, f, indent=4, ensure_ascii=False)

## Core operations

In [3]:
def add_movie(title, genre, rating, path=JSON_FILE):
    title, genre = title.strip(), genre.strip()
    if not title or not genre:
        return False, "Title and genre cannot be empty."
    try:
        rating = float(rating)
    except ValueError:
        return False, "Rating must be a number."
    if not 0 <= rating <= 10:
        return False, "Rating must be between 0 and 10."

    movies = load_movies(path)
    if any(m["title"].lower() == title.lower() for m in movies):
        return False, f"'{title}' is already in your list."

    movies.append({"title": title, "genre": genre, "rating": rating})
    save_movies(movies, path)
    return True, f"'{title}' added."

def search_movies(query, path=JSON_FILE):
    """Case-insensitive partial match on title OR genre."""
    q = query.strip().lower()
    return [m for m in load_movies(path) if q in m["title"].lower() or q in m["genre"].lower()]

def print_table(movies):
    if not movies:
        print("  (no movies to show)")
        return
    w_t = max(len("Title"), *(len(m["title"]) for m in movies))
    w_g = max(len("Genre"), *(len(m["genre"]) for m in movies))
    line = f"+-{'-' * w_t}-+-{'-' * w_g}-+--------+"
    print(line)
    print(f"| {'Title':<{w_t}} | {'Genre':<{w_g}} | Rating |")
    print(line)
    for m in movies:
        print(f"| {m['title']:<{w_t}} | {m['genre']:<{w_g}} | {m['rating']:>6.1f} |")
    print(line)

## Menu loop

In [4]:
def main(path=JSON_FILE):
    while True:
        print("\n===== MOVIE TRACKER =====")
        print("1. Add a movie")
        print("2. View all movies")
        print("3. Search by title or genre")
        print("4. Exit")
        choice = input("Choose an option (1-4): ").strip()

        if choice == "1":
            title = input("Title : ")
            genre = input("Genre : ")
            rating = input("Rating (0-10): ")
            print(add_movie(title, genre, rating, path)[1])
        elif choice == "2":
            print_table(load_movies(path))
        elif choice == "3":
            q = input("Search for: ")
            results = search_movies(q, path)
            print(f"{len(results)} result(s) for '{q}':")
            print_table(results)
        elif choice == "4":
            print("Goodbye!")
            break
        else:
            print("Invalid option, please choose 1-4.")

## Demo (scripted input)
Works on a temporary file `demo_movies.json`. It adds four movies, tries a duplicate and an out-of-range rating, lists them, then searches by a genre keyword and by a title keyword.

In [5]:
from unittest.mock import patch

def run_with_inputs(func, inputs, *args):
    it = iter(inputs)
    def fake_input(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    with patch("builtins.input", fake_input):
        return func(*args)

if os.path.exists("demo_movies.json"):
    os.remove("demo_movies.json")

run_with_inputs(main, [
    "1", "Inception", "Sci-Fi", "8.8",
    "1", "The Dark Knight", "Action", "9.0",
    "1", "Interstellar", "Sci-Fi", "8.7",
    "1", "3 Idiots", "Comedy", "8.4",
    "1", "inception", "Sci-Fi", "9",          # duplicate title
    "1", "Bad Movie", "Drama", "15",          # rating out of range
    "2",
    "3", "sci",                               # genre search
    "3", "dark",                              # title search
    "3", "horror",                            # no match
    "4",
], "demo_movies.json")


===== MOVIE TRACKER =====
1. Add a movie
2. View all movies
3. Search by title or genre
4. Exit
Choose an option (1-4): 1
Title : Inception
Genre : Sci-Fi
Rating (0-10): 8.8
'Inception' added.

===== MOVIE TRACKER =====
1. Add a movie
2. View all movies
3. Search by title or genre
4. Exit
Choose an option (1-4): 1
Title : The Dark Knight
Genre : Action
Rating (0-10): 9.0
'The Dark Knight' added.

===== MOVIE TRACKER =====
1. Add a movie
2. View all movies
3. Search by title or genre
4. Exit
Choose an option (1-4): 1
Title : Interstellar
Genre : Sci-Fi
Rating (0-10): 8.7
'Interstellar' added.

===== MOVIE TRACKER =====
1. Add a movie
2. View all movies
3. Search by title or genre
4. Exit
Choose an option (1-4): 1
Title : 3 Idiots
Genre : Comedy
Rating (0-10): 8.4
'3 Idiots' added.

===== MOVIE TRACKER =====
1. Add a movie
2. View all movies
3. Search by title or genre
4. Exit
Choose an option (1-4): 1
Title : inception
Genre : Sci-Fi
Rating (0-10): 9
'inception' is already in your list

### The JSON file produced

In [6]:
print(open("demo_movies.json").read())

[
    {
        "title": "Inception",
        "genre": "Sci-Fi",
        "rating": 8.8
    },
    {
        "title": "The Dark Knight",
        "genre": "Action",
        "rating": 9.0
    },
    {
        "title": "Interstellar",
        "genre": "Sci-Fi",
        "rating": 8.7
    },
    {
        "title": "3 Idiots",
        "genre": "Comedy",
        "rating": 8.4
    }
]


In [7]:
os.remove("demo_movies.json")
print("demo_movies.json removed")

demo_movies.json removed


## Run it yourself (uses the real `movies.json`)

In [8]:
# main()